# Day 35 — Python + SQLite: Building a Mini Data Warehouse
**Estimated time:** 2 hours  
**Learning objectives:**
1. Design a star schema from the 5 flat CSV sources
2. Load data into proper fact and dimension tables with conformed dimensions
3. Compare flat-file queries vs star schema queries for readability and speed

---
> **SAP Context:** This notebook mirrors the BW/4HANA modelling you do as a data architect. InfoCubes use a star schema (fact table + SID tables for each characteristic). CompositeProviders and Virtual Providers flatten hierarchies. The star schema you build here maps directly to how you'd model these same KPIs in BW InfoCubes or HANA Calculation Views.


## 0. Star Schema Design

### Schema Design
```
                ┌─────────────────────────────────────────────┐
                │                  FACT TABLES                  │
                │                                              │
                │  fact_sales_kpis     fact_inventory           │
                │  (from bw_kpis)      (from materials+orders)  │
                └──────────┬───────────────────┬───────────────┘
                           │                   │
              ┌────────────┼────────────────── ┼────────────┐
              │            │     DIMENSIONS     │            │
              │            │                   │            │
         dim_material   dim_customer        dim_costcenter  dim_time
         (MATNR)        (KUNNR)             (KOSTL)         (YYYYMM)
              │            │                   │            │
         dim_plant      dim_salesorg        dim_employee
         (WERKS)        (VKORG)             (PERNR)
```
**Conformed dimensions:** `dim_material` (MATNR) is shared between `fact_sales_kpis` and `fact_inventory`. `dim_costcenter` (KOSTL) is shared between HR and CO facts.


## 1. Load and Build Dimension Tables

In [ ]:
import sqlite3, pandas as pd, numpy as np, time
from pathlib import Path

from analytics_bootcamp.config import RAW_DATA_DIR as DATA
# Raw data
mat_raw = pd.read_csv(DATA / 'materials_inventory.csv')
so_raw = pd.read_csv(DATA / 'sales_orders.csv')
cc_raw = pd.read_csv(DATA / 'cost_center_actuals.csv')
hr_raw = pd.read_csv(DATA / 'hr_headcount.csv')
bw_raw = pd.read_csv(DATA / 'bw_sales_kpis.csv')

# Connect to star schema DB
star_con = sqlite3.connect(':memory:')

def q(sql, con=star_con): return pd.read_sql_query(sql, con)

# ── dim_material ────────────────────────────────────────────────
dim_material = (
    mat_raw[['MATNR','MAKTX','MATERIAL_TYPE','MRPTYPE']]
    .drop_duplicates('MATNR')
    .reset_index(drop=True)
    .rename(columns={'MAKTX':'MATERIAL_DESC'})
)
dim_material['material_sk'] = dim_material.index + 1  # surrogate key
dim_material.to_sql('dim_material', star_con, if_exists='replace', index=False)

# ── dim_plant ────────────────────────────────────────────────────
dim_plant = (
    mat_raw[['WERKS']].drop_duplicates()
    .assign(PLANT_NAME=lambda d: 'Plant ' + d['WERKS'].astype(str))
    .reset_index(drop=True)
)
dim_plant['plant_sk'] = dim_plant.index + 1
dim_plant.to_sql('dim_plant', star_con, if_exists='replace', index=False)

# ── dim_customer ─────────────────────────────────────────────────
dim_customer = (
    so_raw[['KUNNR']].drop_duplicates()
    .assign(CUSTOMER_NAME=lambda d: 'Customer ' + d['KUNNR'])
    .reset_index(drop=True)
)
dim_customer['customer_sk'] = dim_customer.index + 1
dim_customer.to_sql('dim_customer', star_con, if_exists='replace', index=False)

# ── dim_costcenter ───────────────────────────────────────────────
dim_costcenter = (
    cc_raw[['KOSTL','KTEXT']].drop_duplicates('KOSTL')
    .rename(columns={'KTEXT':'CC_DESC'})
    .reset_index(drop=True)
)
dim_costcenter['cc_sk'] = dim_costcenter.index + 1
dim_costcenter.to_sql('dim_costcenter', star_con, if_exists='replace', index=False)

# ── dim_time ─────────────────────────────────────────────────────
all_periods = pd.Series(
    list(bw_raw['CAL_YEAR_MONTH'].unique()) +
    list(cc_raw['GJAHR'].astype(str).str.cat(cc_raw['PERIOD'].astype(str).str.zfill(2)).astype(int).unique())
).drop_duplicates().sort_values()

dim_time = pd.DataFrame({'CAL_YEAR_MONTH': all_periods})
dim_time['YEAR'] = (dim_time['CAL_YEAR_MONTH'] // 100).astype(int)
dim_time['MONTH'] = (dim_time['CAL_YEAR_MONTH'] % 100).astype(int)
dim_time['QUARTER'] = ((dim_time['MONTH'] - 1) // 3 + 1).astype(int)
dim_time['HALF_YEAR'] = (dim_time['MONTH'] > 6).map({True: 'H2', False: 'H1'})
dim_time['time_sk'] = range(1, len(dim_time) + 1)
dim_time.to_sql('dim_time', star_con, if_exists='replace', index=False)

print('Dimension tables created:')
for tbl in ['dim_material','dim_plant','dim_customer','dim_costcenter','dim_time']:
    n = q(f'SELECT COUNT(*) AS n FROM {tbl}').iloc[0]['n']
    print(f'  {tbl}: {n} rows')

## 2. Build Fact Tables

In [ ]:
# YOUR SOLUTION

# ── fact_sales_kpis (from BW extract) ───────────────────────────
# Join to dimension surrogate keys
fact_kpis = bw_raw.merge(
    dim_material[['MATNR','material_sk']], on='MATNR', how='left'
).merge(
    dim_customer[['KUNNR','customer_sk']], on='KUNNR', how='left'
).merge(
    dim_time[['CAL_YEAR_MONTH','time_sk']], on='CAL_YEAR_MONTH', how='left'
)

fact_kpis_cols = ['time_sk','material_sk','customer_sk',
                  'VKORG','VTWEG','SPART','REGION',
                  'REVENUE','QUANTITY','RETURNS_QTY','RETURNS_VALUE',
                  'COGS','GROSS_MARGIN','DISCOUNT_AMT','NUM_ORDERS']
fact_kpis[fact_kpis_cols].to_sql('fact_sales_kpis', star_con, if_exists='replace', index=False)

# ── fact_inventory ────────────────────────────────────────────────
fact_inventory = mat_raw.merge(dim_material[['MATNR','material_sk']], on='MATNR', how='left')
fact_inventory['INV_VALUE'] = fact_inventory['LABST'] * fact_inventory['STPRS']
fact_inv_cols = ['material_sk','WERKS','LABST','STPRS','INV_VALUE',
                 'LAST_MOVEMENT_DATE','LGORT']
fact_inventory[fact_inv_cols].to_sql('fact_inventory', star_con, if_exists='replace', index=False)

# ── fact_cost_actuals ─────────────────────────────────────────────
fact_cc = cc_raw.merge(dim_costcenter[['KOSTL','cc_sk']], on='KOSTL', how='left')
fact_cc['time_ym'] = fact_cc['GJAHR'] * 100 + fact_cc['PERIOD']
fact_cc = fact_cc.merge(dim_time[['CAL_YEAR_MONTH','time_sk']], left_on='time_ym', right_on='CAL_YEAR_MONTH', how='left')
fact_cc[['cc_sk','time_sk','KSTAR','KSTAR_DESC','ACTUAL_AMT','PLAN_AMT']].to_sql(
    'fact_cost_actuals', star_con, if_exists='replace', index=False)

print('Fact tables created:')
for tbl in ['fact_sales_kpis','fact_inventory','fact_cost_actuals']:
    n = q(f'SELECT COUNT(*) AS n FROM {tbl}').iloc[0]['n']
    print(f'  {tbl}: {n} rows')

## 3. Analytical Queries Against the Star Schema

In [ ]:
# YOUR SOLUTION

# Query 1: Revenue by Year and Region — clean star schema join
print('=== Q1: Revenue by Year and Region ===')
q('''
SELECT
    t.YEAR,
    k.REGION,
    ROUND(SUM(k.REVENUE), 0) AS total_revenue,
    ROUND(SUM(k.GROSS_MARGIN), 0) AS total_gm,
    ROUND(SUM(k.GROSS_MARGIN) / NULLIF(SUM(k.REVENUE),0) * 100, 1) AS gm_pct
FROM fact_sales_kpis k
JOIN dim_time t ON k.time_sk = t.time_sk
GROUP BY t.YEAR, k.REGION
ORDER BY t.YEAR, total_revenue DESC
''').pipe(print)

print()

# Query 2: Top 5 materials by inventory value with material description
print('=== Q2: Top 5 Materials by Inventory Value ===')
q('''
SELECT
    m.MATNR,
    m.MATERIAL_DESC,
    m.MATERIAL_TYPE,
    f.WERKS,
    ROUND(SUM(f.INV_VALUE), 0) AS total_inv_value
FROM fact_inventory f
JOIN dim_material m ON f.material_sk = m.material_sk
GROUP BY m.MATNR, m.MATERIAL_DESC, m.MATERIAL_TYPE, f.WERKS
ORDER BY total_inv_value DESC
LIMIT 5
''').pipe(print)

print()

# Query 3: YTD cost variance — star schema version
print('=== Q3: YTD Cost Variance by Cost Center (2024) ===')
q('''
SELECT
    dc.KOSTL,
    dc.CC_DESC,
    ROUND(SUM(f.ACTUAL_AMT), 0) AS ytd_actual,
    ROUND(SUM(f.PLAN_AMT), 0) AS ytd_plan,
    ROUND(SUM(f.ACTUAL_AMT - f.PLAN_AMT), 0) AS variance
FROM fact_cost_actuals f
JOIN dim_costcenter dc ON f.cc_sk = dc.cc_sk
JOIN dim_time t ON f.time_sk = t.time_sk
WHERE t.YEAR = 2024 AND t.MONTH <= 9
GROUP BY dc.KOSTL, dc.CC_DESC
ORDER BY variance DESC
LIMIT 10
''').pipe(print)

## 4. Compare: Flat File Query vs Star Schema — Readability and Performance

In [ ]:
# YOUR SOLUTION
# Load flat data into a separate flat_con for comparison
flat_con = sqlite3.connect(':memory:')
for tname, raw in [('bw_kpis',bw_raw),('materials',mat_raw),
                   ('cost_centers',cc_raw),('hr',hr_raw),('sales_orders',so_raw)]:
    raw.to_sql(tname, flat_con, if_exists='replace', index=False)

# --- FLAT QUERY ---
flat_sql = '''
SELECT
    (b.CAL_YEAR_MONTH / 100) AS year,
    b.REGION,
    ROUND(SUM(b.REVENUE), 0) AS revenue
FROM bw_kpis b
GROUP BY year, b.REGION
ORDER BY year, revenue DESC
'''

# --- STAR SCHEMA QUERY ---
star_sql = '''
SELECT
    t.YEAR,
    k.REGION,
    ROUND(SUM(k.REVENUE), 0) AS revenue
FROM fact_sales_kpis k
JOIN dim_time t ON k.time_sk = t.time_sk
GROUP BY t.YEAR, k.REGION
ORDER BY t.YEAR, revenue DESC
'''

# Benchmark both
RUNS = 100

start = time.perf_counter()
for _ in range(RUNS):
    pd.read_sql_query(flat_sql, flat_con)
t_flat = (time.perf_counter() - start) / RUNS

start = time.perf_counter()
for _ in range(RUNS):
    pd.read_sql_query(star_sql, star_con)
t_star = (time.perf_counter() - start) / RUNS

print(f'Flat query avg:   {t_flat*1000:.2f} ms')
print(f'Star query avg:   {t_star*1000:.2f} ms')
print()
print("""Key insight: At this data size, the difference is minimal.
Star schema benefits emerge at:
  • Very large fact tables (millions of rows) where dimension filters reduce scan size
  • Complex multi-fact queries that would require large self-joins in flat model
  • Shared dimension logic (single DIM table used by multiple fact tables)
  
In SAP BW: InfoCube = star schema; DSO = flat staging area.
Star schema = optimised for OLAP (aggregation); Flat = optimised for OLTP (row access).""")

## Daily Prompt
**Answer independently:**

1. Add a `dim_salesorg` dimension table with VKORG as the natural key and descriptive columns for sales org name and region. Modify `fact_sales_kpis` to include a `salesorg_sk` surrogate key.
2. In SAP BW InfoCubes, there is a concept of "compressed" fact tables (F-table vs E-table). What is the difference, and how does it relate to the performance trade-offs you observed in this notebook?
3. Write a query that demonstrates the "fan-out" problem: join `fact_sales_kpis` to `fact_cost_actuals` at the wrong grain (MATNR level) and show why the revenue total is inflated. Then fix it using proper aggregation before joining.
